# Mufahras: search by meaning, model comparison

Embeds every verse (Saheeh International) and every English hadith of the six books with several small embedding
models, then scores each against `eval/meaning-queries.json`: does the expected verse or hadith appear in the top 10?

It needs nothing uploaded: the texts come from the live app and the queries from GitHub. Run all cells on a GPU
instance; with a small GPU each small model takes about a minute, the larger reference model a few minutes.

The app will run the chosen model in the browser as an 8-bit ONNX file (Transformers.js), so the sizes below are
those files. The GPU runs here use full precision; the 8-bit version ranks almost identically, which the CPU runs
of three of these models check.

In [ ]:
%pip install -q -U sentence-transformers pandas
import torch; print(torch.__version__, torch.cuda.is_available() and torch.cuda.get_device_name(0))

## The texts and the queries

In [ ]:
import json, re, urllib.request, numpy as np, pandas as pd, time, os
BASE = 'https://omeiirr.github.io/mufahras/data/'
REPO = 'https://raw.githubusercontent.com/omeiirr/mufahras/main/'
get = lambda url: json.load(urllib.request.urlopen(url))
manifest = get(BASE + 'manifest.json')['chunks']
chunk = lambda name: get(BASE + manifest[name]['file'])

# Same texts as the app: footnote markers removed from the translation.
clean = lambda t: re.sub(r'\s+', ' ', re.sub('[\u00B9\u00B2\u00B3\u2070-\u209F]', '', t)).strip()
refs, texts = [], []
quran = chunk('quran')
for s in quran['surahs']:
    for a in range(1, s['count'] + 1):
        refs.append(f"{s['n']}:{a}"); texts.append(clean(quran['en'][s['start'] + a - 1]))
for book in ['bukhari', 'muslim', 'abudawud', 'tirmidhi', 'nasai', 'ibnmajah']:
    for h in chunk(f'hadith-{book}')['hadiths']:
        if h[2] and h[2].strip():
            refs.append(f'{book} {h[0]}'); texts.append(clean(h[2]))
queries = get(REPO + 'eval/meaning-queries.json')
try:
    keyword = get(REPO + 'eval/keyword-baseline.json')   # top 50 from today's keyword search
except Exception:
    keyword = None
print(len(texts), 'texts,', len(queries), 'queries,', 'keyword baseline' if keyword else 'no keyword baseline')

## The candidates

All produce 384-number vectors except the reference. `onnx_mb` is the 8-bit browser download.
`bge-base` is too big for the app; it's here to show how much a larger model would gain.

In [ ]:
BGE_Q = 'Represent this sentence for searching relevant passages: '
MODELS = {
    # key: (Hugging Face id, query prefix, document prefix, 8-bit ONNX size in MB)
    'minilm':      ('sentence-transformers/all-MiniLM-L6-v2', '', '', 23),
    'arctic-xs':   ('Snowflake/snowflake-arctic-embed-xs', BGE_Q, '', 23),
    'mxbai-xs':    ('mixedbread-ai/mxbai-embed-xsmall-v1', BGE_Q, '', 24),
    'bge-small':   ('BAAI/bge-small-en-v1.5', BGE_Q, '', 34),
    'arctic-s':    ('Snowflake/snowflake-arctic-embed-s', BGE_Q, '', 34),
    'e5-small':    ('intfloat/e5-small-v2', 'query: ', 'passage: ', 34),
    'gte-small':   ('thenlper/gte-small', '', '', 34),
    'bge-base':    ('BAAI/bge-base-en-v1.5', BGE_Q, '', 110),   # reference only
}
MAX_TOKENS = 256   # covers 99% of the texts; the app would use the same

## Scoring

In [ ]:
K = 10
def score(ranked):
    """ranked: query -> list of refs, best first. Returns overall numbers and hits per kind."""
    hit = rec = mrr = 0; kinds = {}
    for q in queries:
        exp = list(q['expect']); top = ranked[q['query']][:K]
        found = sum(r in top for r in exp)
        first = next((i for i, r in enumerate(top) if r in exp), None)
        hit += found > 0; rec += found / len(exp); mrr += 0 if first is None else 1 / (first + 1)
        k = kinds.setdefault(q['kind'], [0, 0]); k[0] += found > 0; k[1] += 1
    n = len(queries)
    return {'hit@10': round(100 * hit / n), 'recall@10': round(100 * rec / n), 'MRR': round(mrr / n, 2),
            **{kind: f'{h}/{t}' for kind, (h, t) in kinds.items()}}

def rrf(a, b):
    """Reciprocal-rank fusion of two rankings: the simple blend the app would start with."""
    s = {}
    for lst in (a, b):
        for i, r in enumerate(lst): s[r] = s.get(r, 0) + 1 / (60 + i)
    return sorted(s, key=s.get, reverse=True)

def int8(v):
    """8-bit copy of the vectors with one global scale, as the app would download them."""
    return np.round(v * 127 / np.abs(v).max()).astype(np.int8)

rows = [] if keyword is None else [{'model': 'keyword (today)', **score(keyword)}]
ranked_by = {}

## Run every model

In [ ]:
from sentence_transformers import SentenceTransformer
os.makedirs('vectors', exist_ok=True)
for key, (mid, qp, dp, mb) in MODELS.items():
    model = SentenceTransformer(mid, device='cuda')
    model.max_seq_length = MAX_TOKENS
    t = time.time()
    docs = model.encode([dp + x for x in texts], batch_size=256, normalize_embeddings=True,
                        convert_to_numpy=True, show_progress_bar=False).astype(np.float32)
    secs = time.time() - t
    qv = model.encode([qp + q['query'] for q in queries], normalize_embeddings=True, convert_to_numpy=True)
    np.save(f'vectors/{key}.npy', docs)

    sims = qv @ docs.T
    top = np.argsort(-sims, axis=1)[:, :50]
    sem = {q['query']: [refs[i] for i in top[j]] for j, q in enumerate(queries)}
    d8, q8 = int8(docs), np.clip(np.round(qv * 127 / np.abs(docs).max()), -127, 127).astype(np.int32)
    top8 = np.argsort(-(q8 @ d8.T.astype(np.int32)), axis=1)[:, :50]
    sem8 = {q['query']: [refs[i] for i in top8[j]] for j, q in enumerate(queries)}
    ranked_by[key] = sem

    base = {'onnx_mb': mb, 'dims': docs.shape[1], 'gpu_secs': round(secs)}
    rows.append({'model': key, **base, **score(sem)})
    rows.append({'model': f'{key} int8', **score(sem8)})
    if keyword: rows.append({'model': f'{key} + keyword', **score({q: rrf(keyword[q], sem[q]) for q in sem})})
    print(f'{key}: {secs:.0f}s', score(sem))
    del model; torch.cuda.empty_cache()

results = pd.DataFrame(rows).set_index('model')
results

## Where each model misses

For each query: the rank (1–10) of the first expected text, or `-` when none is in the top 10.

In [ ]:
miss = pd.DataFrame({key: [next((str(i + 1) for i, r in enumerate(sem[q['query']][:K]) if r in q['expect']), '-')
                           for q in queries] for key, sem in ranked_by.items()},
                    index=[f"{q['kind']}: {q['query']}" for q in queries])
pd.set_option('display.max_rows', 100); pd.set_option('display.max_colwidth', 70)
miss

## What the best model returns for a query

In [ ]:
def show(query, key='bge-small', n=10):
    q = next(x for x in queries if x['query'] == query)
    for r in ranked_by[key][query][:n]:
        print('*' if r in q['expect'] else ' ', r.ljust(14), texts[refs.index(r)][:110])
    print('expected:', ', '.join(q['expect']))
show(queries[0]['query'])

## Save the results

Download `meaning-spike-results.zip` (the table, the misses and the vectors) and send me the two CSVs.

In [ ]:
results.to_csv('results.csv'); miss.to_csv('misses.csv')
!zip -qr meaning-spike-results.zip results.csv misses.csv vectors && ls -lh meaning-spike-results.zip results.csv misses.csv